# 09 — Strategy performance **now**

Read-only snapshot: **frozen TEMA OOS** (baseline doc) vs **fresh Vision re-run** on the default 10 symbols, **live journal**, and **Donchian turtle** research book.

Does not change live `W_*` or `.env`. First Vision pass may download/cache months (~minutes).

In [ ]:
import sys
from pathlib import Path
from datetime import date
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif (ROOT / "python" / "research").exists():
    ROOT = ROOT / "python"
sys.path.insert(0, str(ROOT))

from research.strategy_performance import (
    FROZEN_BASELINE,
    GOALS,
    default_window,
    goals_check,
    journal_stats_sync,
    run_tema_backtest,
    DEFAULT_SYMBOLS,
)
from backtest.donchian_turtle import (
    apply_split_metrics,
    resolve_universe,
    run_turtle_portfolio,
)
from scanner.donchian_turtle import TurtleParams

start, end, split = default_window()
print("Window:", start, "→", end, "| OOS from", split)


## 1. Frozen baseline (canonical OOS)

In [ ]:
baseline_rows = [
    {"book": "Combined 1h+4h A/A+", **FROZEN_BASELINE["combined_aa_oos"]},
    {"book": "4h A/A+ (edge slice)", **FROZEN_BASELINE["four_h_aa_oos"]},
    {"book": "1h A/A+", **FROZEN_BASELINE["one_h_aa_oos"]},
]
baseline_df = pd.DataFrame(baseline_rows)
print(baseline_df.to_string(index=False))
print(FROZEN_BASELINE["proposed_knob"])


## 2. Fresh TEMA backtest (Vision, canonical filters)

In [ ]:
frame, summaries = run_tema_backtest(
    DEFAULT_SYMBOLS,
    ["4h", "1h"],
    start,
    end,
    split=split,
)
refresh_rows = []
for key, s in summaries.items():
    if s is None:
        continue
    row = s.to_dict()
    row["passes_goals"] = goals_check(s)
    refresh_rows.append(row)
refresh_df = pd.DataFrame(refresh_rows)
print(refresh_df.to_string(index=False))
print("Signals in frame:", len(frame), "| grades A/A+:", len(frame[frame['grade'].isin(['A','A+'])]) if len(frame) else 0)


## 3. Live journal (desk fills — not OOS)

In [ ]:
journal = journal_stats_sync()
journal


## 4. Donchian turtle (research module, top-30 universe smoke)

In [ ]:
uni, src = resolve_universe(30)
turtle = run_turtle_portfolio(
    uni, start, end, params=TurtleParams(), max_positions=10
)
apply_split_metrics(turtle, split)
print("universe", src, "loaded", len(turtle.universe))
print("full", turtle.metrics)
print("OOS", turtle.oos_metrics)


## 5. Verdict vs `strategy/goals.yaml`

In [ ]:
s4 = summaries.get("4h")
if s4:
    print("4h OOS refresh vs goals:", goals_check(s4), "| goals:", GOALS)
    print("Compare to frozen 4h OOS:", FROZEN_BASELINE["four_h_aa_oos"])
else:
    print("No 4h OOS rows — check Vision cache / window")
